In [1]:
from pathlib import Path
from dmpbridge.evaluation.pdfplumber_text_evaluator import evaluate_pdfplumber_text

result = evaluate_pdfplumber_text(
    extracted_txt_path=Path("../data/pdfplumber_extracted_text/sample4.txt"),
    reference_txt_path=Path("../data/reference_text/sample4_reference.txt"),
)

result

{'sample_id': 'sample4',
 'word_capture': 0.999,
 'rouge_l': 1.0,
 'extracted_word_count': 1016,
 'reference_word_count': 1013,
 'missing_word_count': 1,
 'extra_word_count': 4,
 'missing_words_preview': 'platformindependent',
 'extra_words_preview': 'page, page, platform, independent',
 'extracted_line_count': 86,
 'reference_line_count': 78}

In [5]:
import pandas as pd
from pathlib import Path

from dmpbridge.evaluation.pdfplumber_text_evaluator import evaluate_pdfplumber_text

results = []

extracted_folder = Path("../data/pdfplumber_extracted_text")
reference_folder = Path("../data/reference_text")

for extracted_file in sorted(extracted_folder.glob("*.txt")):

    sample_id = extracted_file.stem
    reference_file = reference_folder / f"{sample_id}_reference.txt"

    if not reference_file.exists():
        print(f"Missing reference file: {reference_file}")
        continue

    result = evaluate_pdfplumber_text(
        extracted_txt_path=extracted_file,
        reference_txt_path=reference_file,
    )

    results.append(result)

df = pd.DataFrame(results)

display(df)

,sample_id,word_capture,rouge_l,extracted_word_count,reference_word_count,missing_word_count,extra_word_count,missing_words_preview,extra_words_preview,extracted_line_count,reference_line_count
0,sample1,1.000,0.996,990,988,0,2,,"page, page",87,40
1,sample10,0.993,0.999,885,886,6,5,"management, and, cdl, also, subject, after","page, page, andmanagement, cdlalso, subjectafter",76,68
2,sample2,0.999,1.000,2176,2172,2,6,"f, rom","page, page, page, page, page, from",191,71
3,sample3,1.000,0.236,1133,851,0,282,,"page, page, page, cps, roles, roles, and, and,...",81,69
4,sample4,0.999,1.000,1016,1013,1,4,platformindependent,"page, page, platform, independent",86,78
5,sample5,0.999,0.997,1060,1057,1,4,recordlevel,"page, page, record, level",89,80
6,sample6,0.996,1.000,279,277,1,3,opensource,"page, open, source",28,24
7,sample7,1.000,1.000,263,262,0,1,,page,21,17
8,sample8,1.000,1.000,831,829,0,2,,"page, page",67,59
9,sample9,0.999,0.979,1162,1120,1,43,publiclyavailable,"page, page, page, career, high, resolution, nm...",97,85


In [6]:
from pathlib import Path
import re
import time
import difflib
import pandas as pd

extracted_folder = Path("../data/pdfplumber_extracted_text")
reference_folder = Path("../data/reference_text")

output_folder = Path("../outputs/evaluation")
output_folder.mkdir(parents=True, exist_ok=True)

report_path = output_folder / "pdfplumber_paper_style_metrics.csv"


def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_paragraphs(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def tokenize_words(text):
    text = text.lower()
    text = re.sub(r"[^\w\s'-]", " ", text)
    return re.findall(r"\b\w+(?:[-']\w+)?\b", text)


def safe_percent(value, denominator):
    return 0.0 if denominator == 0 else round((value / denominator) * 100, 2)


def get_extracted_filename(reference_path):
    return reference_path.name.replace("_reference", "")


def word_diff_metrics(reference_words, extracted_words):
    matcher = difflib.SequenceMatcher(None, reference_words, extracted_words)

    w_plus = 0
    w_minus = 0
    w_tilde = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        ref_chunk = reference_words[i1:i2]
        ext_chunk = extracted_words[j1:j2]

        if tag == "insert":
            w_plus += len(ext_chunk)

        elif tag == "delete":
            w_minus += len(ref_chunk)

        elif tag == "replace":
            min_len = min(len(ref_chunk), len(ext_chunk))

            for k in range(min_len):
                similarity = difflib.SequenceMatcher(
                    None, ref_chunk[k], ext_chunk[k]
                ).ratio()

                if similarity >= 0.75:
                    w_tilde += 1
                else:
                    w_minus += 1
                    w_plus += 1

            w_minus += max(0, len(ref_chunk) - min_len)
            w_plus += max(0, len(ext_chunk) - min_len)

    return w_plus, w_minus, w_tilde


def newline_metrics(reference_text, extracted_text):
    reference_newlines = reference_text.count("\n")
    extracted_newlines = extracted_text.count("\n")

    nl_plus = max(0, extracted_newlines - reference_newlines)
    nl_minus = max(0, reference_newlines - extracted_newlines)

    return nl_plus, nl_minus, reference_newlines, extracted_newlines


def normalize_paragraph_for_match(paragraph):
    return " ".join(tokenize_words(paragraph))


def paragraph_similarity(p1, p2):
    return difflib.SequenceMatcher(
        None,
        normalize_paragraph_for_match(p1),
        normalize_paragraph_for_match(p2),
    ).ratio()


def paragraph_metrics(reference_paragraphs, extracted_paragraphs, similarity_threshold=0.70):
    matched_reference = set()
    matched_extracted = set()
    matches = []

    for ext_i, ext_p in enumerate(extracted_paragraphs):
        best_ref_i = None
        best_score = 0

        for ref_i, ref_p in enumerate(reference_paragraphs):
            if ref_i in matched_reference:
                continue

            score = paragraph_similarity(ref_p, ext_p)

            if score > best_score:
                best_score = score
                best_ref_i = ref_i

        if best_ref_i is not None and best_score >= similarity_threshold:
            matched_reference.add(best_ref_i)
            matched_extracted.add(ext_i)
            matches.append((ext_i, best_ref_i, best_score))

    p_plus = len(extracted_paragraphs) - len(matched_extracted)
    p_minus = len(reference_paragraphs) - len(matched_reference)

    sorted_matches = sorted(matches, key=lambda x: x[0])
    reference_order = [ref_i for _, ref_i, _ in sorted_matches]

    p_reordered = 0
    for i in range(1, len(reference_order)):
        if reference_order[i] < reference_order[i - 1]:
            p_reordered += 1

    return p_plus, p_minus, p_reordered


def compute_z_score(
    nl_plus, nl_minus,
    w_plus, w_minus, w_tilde,
    p_plus, p_minus, p_reordered,
    c=5
):
    return (
        nl_plus + nl_minus
        + w_plus + w_minus + w_tilde
        + c * (p_plus + p_minus + p_reordered)
    )


def identify_main_problem(nl_plus, nl_minus, p_plus, p_minus, p_reordered, w_plus, w_minus):
    problems = {
        "extra_newlines": nl_plus,
        "missing_newlines": nl_minus,
        "extra_paragraphs": p_plus * 5,
        "missing_paragraphs": p_minus * 5,
        "reordered_paragraphs": p_reordered * 5,
        "extra_words": w_plus,
        "missing_words": w_minus,
    }

    main_problem = max(problems, key=problems.get)

    if problems[main_problem] == 0:
        return "No major issue"

    return main_problem


def overall_quality(z_score, gt_words):
    if gt_words == 0:
        return "Unknown"

    normalized = z_score / gt_words

    if normalized <= 0.03:
        return "Excellent"
    elif normalized <= 0.08:
        return "Good"
    elif normalized <= 0.15:
        return "Fair"
    else:
        return "Poor"


def evaluate_file(reference_path, extracted_path):
    start_time = time.time()

    reference_text = normalize_text(
        reference_path.read_text(encoding="utf-8", errors="ignore")
    )

    extracted_text = normalize_text(
        extracted_path.read_text(encoding="utf-8", errors="ignore")
    )

    runtime = time.time() - start_time

    reference_words = tokenize_words(reference_text)
    extracted_words = tokenize_words(extracted_text)

    reference_paragraphs = split_paragraphs(reference_text)
    extracted_paragraphs = split_paragraphs(extracted_text)

    gt_words = len(reference_words)
    extracted_word_count = len(extracted_words)

    nl_plus, nl_minus, gt_newlines, extracted_newlines = newline_metrics(
        reference_text, extracted_text
    )

    p_plus, p_minus, p_reordered = paragraph_metrics(
        reference_paragraphs, extracted_paragraphs
    )

    w_plus, w_minus, w_tilde = word_diff_metrics(
        reference_words, extracted_words
    )

    z_score = compute_z_score(
        nl_plus, nl_minus,
        w_plus, w_minus, w_tilde,
        p_plus, p_minus, p_reordered,
        c=5
    )

    word_capture_percent = safe_percent(gt_words - w_minus, gt_words)

    return {
        "file_name": reference_path.stem,
        "reference_file": reference_path.name,
        "extracted_file": extracted_path.name,

        "GT_words": gt_words,
        "Extracted_words": extracted_word_count,
        "Word_count_difference": extracted_word_count - gt_words,
        "Word_capture_%": word_capture_percent,

        "GT_newlines": gt_newlines,
        "Extracted_newlines": extracted_newlines,
        "Line_count_difference": extracted_newlines - gt_newlines,

        "GT_paragraphs": len(reference_paragraphs),
        "Extracted_paragraphs": len(extracted_paragraphs),

        "NL+": nl_plus,
        "NL+_%": safe_percent(nl_plus, gt_newlines),
        "NL-": nl_minus,
        "NL-_%": safe_percent(nl_minus, gt_newlines),

        "P+": p_plus,
        "P-": p_minus,
        "P_reordered": p_reordered,

        "W+": w_plus,
        "W+_%": safe_percent(w_plus, gt_words),
        "W-": w_minus,
        "W-_%": safe_percent(w_minus, gt_words),
        "W_misspelled": w_tilde,
        "W_misspelled_%": safe_percent(w_tilde, gt_words),

        "Missing_word_rate_%": safe_percent(w_minus, gt_words),
        "Extra_word_rate_%": safe_percent(w_plus, gt_words),

        "Main_problem": identify_main_problem(
            nl_plus, nl_minus, p_plus, p_minus, p_reordered, w_plus, w_minus
        ),

        "Z_score_c5": z_score,
        "Overall_quality": overall_quality(z_score, gt_words),

        "ERR": 0,
        "T_seconds": round(runtime, 4),
    }


results = []

reference_files = sorted(reference_folder.glob("*.txt"))

for reference_path in reference_files:
    extracted_filename = get_extracted_filename(reference_path)
    extracted_path = extracted_folder / extracted_filename

    print("Reference:", reference_path.name, "-> Extracted:", extracted_filename)

    if not extracted_path.exists():
        results.append({
            "file_name": reference_path.stem,
            "reference_file": reference_path.name,
            "expected_extracted_file": extracted_filename,
            "ERR": 1,
            "error_message": "Missing extracted pdfplumber text file"
        })
        continue

    try:
        results.append(evaluate_file(reference_path, extracted_path))

    except Exception as e:
        results.append({
            "file_name": reference_path.stem,
            "reference_file": reference_path.name,
            "expected_extracted_file": extracted_filename,
            "ERR": 1,
            "error_message": str(e)
        })

df_metrics = pd.DataFrame(results)
df_metrics.to_csv(report_path, index=False)

print("Evaluation completed.")
print("Reference files:", len(reference_files))
print("Saved report to:", report_path)

df_metrics

Reference: sample10_reference.txt -> Extracted: sample10.txt
Reference: sample1_reference.txt -> Extracted: sample1.txt
Reference: sample2_reference.txt -> Extracted: sample2.txt
Reference: sample3_reference.txt -> Extracted: sample3.txt
Reference: sample4_reference.txt -> Extracted: sample4.txt
Reference: sample5_reference.txt -> Extracted: sample5.txt
Reference: sample6_reference.txt -> Extracted: sample6.txt
Reference: sample7_reference.txt -> Extracted: sample7.txt
Reference: sample8_reference.txt -> Extracted: sample8.txt
Reference: sample9_reference.txt -> Extracted: sample9.txt
Evaluation completed.
Reference files: 10
Saved report to: ..\outputs\evaluation\pdfplumber_paper_style_metrics.csv


,file_name,reference_file,extracted_file,GT_words,Extracted_words,Word_count_difference,Word_capture_%,GT_newlines,Extracted_newlines,Line_count_difference,...,W-_%,W_misspelled,W_misspelled_%,Missing_word_rate_%,Extra_word_rate_%,Main_problem,Z_score_c5,Overall_quality,ERR,T_seconds
0,sample10_reference,sample10_reference.txt,sample10.txt,890,891,1,99.33,67,72,5,...,0.67,0,0.00,0.67,0.79,extra_paragraphs,33,Good,0,0.000
1,sample1_reference,sample1_reference.txt,sample1.txt,991,997,6,99.80,39,83,44,...,0.20,0,0.00,0.20,0.81,extra_newlines,79,Good,0,0.001
2,sample2_reference,sample2_reference.txt,sample2.txt,2165,2174,9,99.91,59,184,125,...,0.09,0,0.00,0.09,0.51,extra_newlines,248,Fair,0,0.001
3,sample3_reference,sample3_reference.txt,sample3.txt,848,1132,284,92.45,68,76,8,...,7.55,1,0.12,7.55,41.04,extra_words,456,Poor,0,0.000
4,sample4_reference,sample4_reference.txt,sample4.txt,1015,1020,5,99.90,77,82,5,...,0.10,0,0.00,0.10,0.59,extra_paragraphs,37,Good,0,0.000
5,sample5_reference,sample5_reference.txt,sample5.txt,1072,1077,5,99.91,79,85,6,...,0.09,0,0.00,0.09,0.56,extra_paragraphs,28,Excellent,0,0.001
6,sample6_reference,sample6_reference.txt,sample6.txt,284,287,3,99.65,23,25,2,...,0.35,0,0.00,0.35,1.41,extra_paragraphs,12,Good,0,0.000
7,sample7_reference,sample7_reference.txt,sample7.txt,259,261,2,100.00,16,18,2,...,0.00,0,0.00,0.00,0.77,extra_paragraphs,9,Good,0,0.000
8,sample8_reference,sample8_reference.txt,sample8.txt,835,839,4,100.00,58,63,5,...,0.00,0,0.00,0.00,0.48,extra_paragraphs,24,Excellent,0,0.001
9,sample9_reference,sample9_reference.txt,sample9.txt,1105,1147,42,99.28,84,92,8,...,0.72,0,0.00,0.72,4.52,extra_words,101,Fair,0,0.001


In [7]:
from pathlib import Path
import json
import re
import pandas as pd

json_folder = Path("../data/pdfplumber_extracted_blocks")
reference_folder = Path("../data/reference_text")

output_folder = Path("../outputs/evaluation")
output_folder.mkdir(parents=True, exist_ok=True)

report_path = output_folder / "pdfplumber_json_structure_evaluation.csv"


def load_json_blocks(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # If your JSON is directly a list
    if isinstance(data, list):
        return data

    # If your JSON has key like "lines" or "blocks"
    for key in ["lines", "blocks", "text_blocks"]:
        if key in data:
            return data[key]

    raise ValueError(f"Cannot find blocks/lines in {json_path.name}")


def normalize_text(text):
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_bold_line(block):
    font_name = str(block.get("font_name", "")).lower()
    is_bold = block.get("is_bold", False)

    return (
        is_bold is True
        or "bold" in font_name
        or "black" in font_name
        or "semibold" in font_name
    )


def get_font_size(block):
    for key in ["avg_font_size", "font_size", "size"]:
        if key in block and block[key] is not None:
            try:
                return float(block[key])
            except:
                return 0.0
    return 0.0


def detect_header_like_lines(blocks):
    font_sizes = [get_font_size(b) for b in blocks if get_font_size(b) > 0]

    if font_sizes:
        median_font = sorted(font_sizes)[len(font_sizes) // 2]
    else:
        median_font = 0

    headers = []

    for i, block in enumerate(blocks):
        text = normalize_text(block.get("text", ""))

        if not text:
            continue

        font_size = get_font_size(block)
        bold = is_bold_line(block)

        header_pattern = bool(
            re.search(r"\b(Element|Section|Question|Data Type|Data Sharing|Repository|Access|Metadata)\b", text, re.I)
            or re.match(r"^\d+[\.\)]\s+", text)
            or re.match(r"^[A-Z][A-Za-z\s/,-]{3,80}:?$", text)
        )

        larger_font = font_size > median_font + 1 if median_font else False
        short_line = len(text.split()) <= 12

        if (header_pattern and short_line) or (bold and short_line) or (larger_font and short_line):
            headers.append({
                "line_index": i,
                "page": block.get("page"),
                "line_order": block.get("line_order"),
                "text": text,
                "font_size": font_size,
                "is_bold": bold,
                "reason": {
                    "header_pattern": header_pattern,
                    "larger_font": larger_font,
                    "short_line": short_line
                }
            })

    return headers


def detect_possible_merged_title_answer(blocks):
    flagged = []

    patterns = [
        r"Element\s+\d+[:\-].{60,}",
        r"Section\s+\d+[:\-].{60,}",
        r"Question\s+\d+[:\-].{60,}",
        r"^[A-Z][A-Za-z\s/,-]{3,50}:\s+[A-Z][a-z].{30,}",
    ]

    for i, block in enumerate(blocks):
        text = normalize_text(block.get("text", ""))

        for pat in patterns:
            if re.search(pat, text):
                flagged.append({
                    "line_index": i,
                    "page": block.get("page"),
                    "line_order": block.get("line_order"),
                    "text": text[:300]
                })
                break

    return flagged


def check_line_order(blocks):
    problems = []

    prev_page = None
    prev_order = None

    for i, block in enumerate(blocks):
        page = block.get("page")
        line_order = block.get("line_order")

        if page is None or line_order is None:
            continue

        if prev_page == page and prev_order is not None:
            if line_order < prev_order:
                problems.append({
                    "line_index": i,
                    "page": page,
                    "previous_order": prev_order,
                    "current_order": line_order,
                    "text": normalize_text(block.get("text", ""))[:200]
                })

        prev_page = page
        prev_order = line_order

    return problems


def extract_reference_headers(reference_text):
    headers = []

    for line in reference_text.splitlines():
        line = normalize_text(line)

        if not line:
            continue

        if (
            re.search(r"\b(Element|Section|Question|Data Type|Data Sharing|Repository|Access|Metadata)\b", line, re.I)
            or re.match(r"^\d+[\.\)]\s+", line)
        ):
            headers.append(line)

    return headers


def header_recall(reference_headers, detected_headers):
    detected_text = " ".join(h["text"].lower() for h in detected_headers)

    if not reference_headers:
        return None

    matched = 0

    for ref in reference_headers:
        ref_words = set(re.findall(r"\w+", ref.lower()))
        detected_words = set(re.findall(r"\w+", detected_text))

        if ref_words and len(ref_words & detected_words) / len(ref_words) >= 0.5:
            matched += 1

    return round(matched / len(reference_headers), 3)


def get_json_filename_from_reference(reference_path):
    # sample1_reference.txt -> sample1.json
    return reference_path.name.replace("_reference.txt", ".json")


def evaluate_json_structure(reference_path, json_path):
    blocks = load_json_blocks(json_path)

    reference_text = reference_path.read_text(encoding="utf-8", errors="ignore")
    reference_headers = extract_reference_headers(reference_text)

    detected_headers = detect_header_like_lines(blocks)
    merged_lines = detect_possible_merged_title_answer(blocks)
    order_problems = check_line_order(blocks)

    total_lines = len(blocks)
    bold_lines = sum(1 for b in blocks if is_bold_line(b))
    font_sizes = [get_font_size(b) for b in blocks if get_font_size(b) > 0]

    avg_font_size = round(sum(font_sizes) / len(font_sizes), 2) if font_sizes else 0

    h_recall = header_recall(reference_headers, detected_headers)

    structure_score = 100

    structure_score -= min(len(merged_lines) * 10, 30)
    structure_score -= min(len(order_problems) * 10, 30)

    if h_recall is not None:
        structure_score -= round((1 - h_recall) * 40)

    structure_score = max(0, structure_score)

    return {
        "file_name": reference_path.stem,
        "reference_file": reference_path.name,
        "json_file": json_path.name,

        "total_json_lines": total_lines,
        "bold_lines": bold_lines,
        "avg_font_size": avg_font_size,

        "reference_header_count": len(reference_headers),
        "detected_header_count": len(detected_headers),
        "header_recall": h_recall,

        "possible_merged_title_answer_count": len(merged_lines),
        "line_order_problem_count": len(order_problems),

        "structure_score_0_100": structure_score,

        "detected_headers_preview": " | ".join(h["text"] for h in detected_headers[:5]),
        "merged_line_preview": " | ".join(m["text"] for m in merged_lines[:3]),

        "ERR": 0
    }


results = []

reference_files = sorted(reference_folder.glob("*.txt"))

for reference_path in reference_files:
    json_filename = get_json_filename_from_reference(reference_path)
    json_path = json_folder / json_filename

    print("Reference:", reference_path.name, "-> JSON:", json_filename)

    if not json_path.exists():
        results.append({
            "file_name": reference_path.stem,
            "reference_file": reference_path.name,
            "expected_json_file": json_filename,
            "ERR": 1,
            "error_message": "Missing pdfplumber JSON file"
        })
        continue

    try:
        results.append(evaluate_json_structure(reference_path, json_path))

    except Exception as e:
        results.append({
            "file_name": reference_path.stem,
            "reference_file": reference_path.name,
            "expected_json_file": json_filename,
            "ERR": 1,
            "error_message": str(e)
        })

df_json_eval = pd.DataFrame(results)
df_json_eval.to_csv(report_path, index=False)

print("JSON structure evaluation completed.")
print("Saved report to:", report_path)

df_json_eval

Reference: sample10_reference.txt -> JSON: sample10.json
Reference: sample1_reference.txt -> JSON: sample1.json
Reference: sample2_reference.txt -> JSON: sample2.json
Reference: sample3_reference.txt -> JSON: sample3.json
Reference: sample4_reference.txt -> JSON: sample4.json
Reference: sample5_reference.txt -> JSON: sample5.json
Reference: sample6_reference.txt -> JSON: sample6.json
Reference: sample7_reference.txt -> JSON: sample7.json
Reference: sample8_reference.txt -> JSON: sample8.json
Reference: sample9_reference.txt -> JSON: sample9.json
JSON structure evaluation completed.
Saved report to: ..\outputs\evaluation\pdfplumber_json_structure_evaluation.csv


,file_name,reference_file,json_file,total_json_lines,bold_lines,avg_font_size,reference_header_count,detected_header_count,header_recall,possible_merged_title_answer_count,line_order_problem_count,structure_score_0_100,detected_headers_preview,merged_line_preview,ERR
0,sample10_reference,sample10_reference.txt,sample10.json,68,7,10.98,9,7,0.667,0,0,87,DATA MANAGEMENT | 1. Policy and Practice | 2. ...,,0
1,sample1_reference,sample1_reference.txt,sample1.json,79,16,11.04,21,15,0.476,0,0,79,DATA MANAGEMENT AND SHARING PLAN | Element 1: ...,,0
2,sample2_reference,sample2_reference.txt,sample2.json,171,42,11.25,23,28,0.565,0,0,83,Center for Bio-Inspired Energy Science | 1. Da...,,0
3,sample3_reference,sample3_reference.txt,sample3.json,69,0,11.90,7,8,0.143,0,0,66,CPS CPS 2015 2015 | Roles Roles and and respon...,,0
4,sample4_reference,sample4_reference.txt,sample4.json,78,1,12.00,2,1,0.000,0,0,60,DATA MANAGEMENT PLAN,,0
5,sample5_reference,sample5_reference.txt,sample5.json,81,16,11.04,22,11,0.273,6,0,41,DATA MANAGEMENT PLAN | REVIEW OF PROPOSAL COMP...,Input data: Available data will be systematica...,0
6,sample6_reference,sample6_reference.txt,sample6.json,24,1,12.00,5,1,0.000,0,0,60,Data Management Plan:,,0
7,sample7_reference,sample7_reference.txt,sample7.json,17,1,11.04,3,1,0.333,0,0,73,Resource/Data Sharing Plan,,0
8,sample8_reference,sample8_reference.txt,sample8.json,59,8,9.84,10,7,0.600,0,0,84,"Univ. of California, Riverside Data Management...",,0
9,sample9_reference,sample9_reference.txt,sample9.json,85,0,11.55,4,6,0.250,0,0,70,CAREER: CAREER: HIGH-RESOLUTION HIGH-RESOLUTIO...,,0
